# NuCLS E2 · Cola única reanudable

Usar **después de finalizar la corrida actual**, en GPU T4 y la misma cuenta de Drive. Ejecutar todas las celdas en orden. Conectar Drive al inicio; después no hay preguntas intermedias.

La cola completa búsqueda faltante, perfil/búsqueda de tres nuevos modelos, pilotos de cinco semillas para los diez seleccionados, referencias simples, evaluación exploratoria de tres mejores (robustez, saliencia, estabilidad y latencia), exportación y métricas. Conserva checkpoints y omite entrenamientos completos verificados. No cambia E1 ni abre reserva.

**Límite real:** no ejecuta un benchmark de tamaño aún no fijado, la replicación del artículo en PCam, ni análisis multi-rater sin anotaciones. No acredita inferencia confirmatoria o cobertura conformal independiente. Al terminar enumera estas dependencias; no declara terminado todo E2. Puede durar más de una sesión de Colab. Ante corte, volver a ejecutar: se conserva la última época guardada; el trabajo posterior al último checkpoint puede repetirse. Una marca de otro entorno exige comprobación y no se elimina automáticamente.


In [ ]:
import os,sys,json,platform,datetime,subprocess,zipfile,hashlib,base64,io
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
from google.colab import drive
mount=Path('/content/drive')
if not os.path.ismount(mount):
    if mount.exists() and any(mount.iterdir()):
        raise RuntimeError('La carpeta de montaje contiene archivos pero no está montada. No se movió ni borró nada; requiere revisar conexión.')
    drive.mount(str(mount))
DRIVE=mount/'MyDrive'
ROOT=DRIVE/'NuCLS_E2_COLA_UNICA_V1'
assert not ROOT.is_symlink()
ROOT.mkdir(exist_ok=True)
def comprobar_bloqueo(p):
    if not p.exists():return
    info=json.loads(p.read_text())
    if info.get('host')!=platform.node():raise RuntimeError('Marca de otra sesión: '+str(p)+'. Verificar esa sesión antes de reanudar; no se borró.')
    pid=int(info['pid'])
    if pid<=0:raise RuntimeError('PID de bloqueo inválido')
    try:os.kill(pid,0)
    except ProcessLookupError:
        p.rename(p.with_name(p.name+'.respaldo_'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%f')))
    else:raise RuntimeError('Proceso aún presente: '+str(pid)+'. No iniciar otra copia.')
for folder in ['NuCLS_E2_ADICIONALES_EXPLORATORIO','NuCLS_E2_TRES_NUEVOS_V1','NuCLS_E2_COLA_UNICA_V1']:
    comprobar_bloqueo(DRIVE/folder/'EJECUCION_ACTIVA.json')
print('Drive conectado y bloqueos comprobados. No se inició entrenamiento.')


## Preparar el código original y los datos de desarrollo
Se reutilizan los paquetes originales íntegros para conservar su identidad. Si el entorno cambió, se detiene antes del entrenamiento.

In [ ]:
import os, sys, json, base64, hashlib, zipfile, io, subprocess
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from google.colab import drive

BASES = Path('/content/drive/MyDrive/NuCLS_E2_CONTINUACION_BASES_22')
INDICE = BASES / 'indice_validacion.csv'
if not INDICE.is_file():
    INDICE = BASES / 'final/indice_validacion.csv'
assert INDICE.is_file(), 'No aparece el cierre de 22 semillas: conecta la misma cuenta de Drive.'
CODIGO = Path('/content/nucls_e2_adicionales_codigo')
DATOS = Path('/content/nucls_e2_datos')
SALIDA = Path('/content/drive/MyDrive/NuCLS_E2_ADICIONALES_EXPLORATORIO')
raw = base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest() == '1e26743076b4f2406fde42ffe12c0314e7c699e7737da966d52fb6e5ae23d27e'
CODIGO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for name in z.namelist():
        target = (CODIGO / name).resolve()
        assert target.is_relative_to(CODIGO.resolve())
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(z.read(name))
print('Código verificado. Resultados nuevos:', SALIDA)

CODIGO_ANTERIOR=CODIGO; DATOS_ANTERIORES=DATOS; SALIDA_ANTERIOR=SALIDA
import csv, platform
import torch, torchvision, numpy as np
assert torch.cuda.is_available(), 'Selecciona GPU en Entorno de ejecución → Cambiar tipo.'
filas = list(csv.DictReader(INDICE.open()))
original = next(Path(r['ruta']).parents[1] for r in filas if 'NuCLS_V2_RESULTADOS_pool_corregido' in r['ruta'])
cfg = json.loads((original / 'configuracion.json').read_text())
actual = {'torch':torch.__version__, 'torchvision':torchvision.__version__, 'numpy':np.__version__}
print('Entorno actual:', actual)
for k, v in actual.items():
    assert v == cfg[k], f'Cambió {k}: actual {v}; anterior {cfg[k]}. No mezclar sin documentar.'
assert platform.python_version_tuple()[:2] == tuple(cfg['python'].split()[0].split('.')[:2]), 'Cambió la versión mayor/menor de Python.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'timm==1.0.22'], check=True)
import timm
assert timm.__version__ == '1.0.22', 'Reinicia la sesión si habías importado otra versión de timm.'
print('GPU:', torch.cuda.get_device_name(0), '| timm:', timm.__version__)
if not (DATOS / 'manifiesto.csv').is_file():
    fuente = Path('/content/drive/MyDrive/NuCLS_V2_DATOS.zip')
    assert fuente.is_file(), 'No aparece NuCLS_V2_DATOS.zip en este Drive.'
    subprocess.run([sys.executable, '-u', str(CODIGO/'scripts/preparar_datos.py'), '--zip', str(fuente), '--salida', str(DATOS)], check=True)
print('Preparación lista. El perfil verificará el manifiesto y las imágenes.')


In [ ]:
import os,sys,json,base64,hashlib,zipfile,io,subprocess
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from google.colab import drive

PREVIO=Path('/content/drive/MyDrive/NuCLS_E2_ADICIONALES_EXPLORATORIO')
assert not (PREVIO/'EJECUCION_ACTIVA.json').exists(), 'La búsqueda anterior conserva una marca activa. No iniciar otra ejecución ni borrar ese archivo sin verificar el proceso.'
BASES=Path('/content/drive/MyDrive/NuCLS_E2_CONTINUACION_BASES_22')
INDICE=BASES/'indice_validacion.csv'
if not INDICE.is_file(): INDICE=BASES/'final/indice_validacion.csv'
assert INDICE.is_file(), 'Conecta la cuenta que contiene las 66 corridas base.'
CODIGO=Path('/content/nucls_e2_tres_nuevos_v1_codigo')
DATOS=Path('/content/nucls_e2_tres_nuevos_v1_datos')
SALIDA=Path('/content/drive/MyDrive/NuCLS_E2_TRES_NUEVOS_V1')
assert not (SALIDA/'EJECUCION_ACTIVA.json').exists(), 'Este bloque tiene una marca activa; verificar antes de reanudar.'
raw=base64.b64decode('UEsDBBQAAAAIABN7a102uKvtJQEAAJ8BAAAZAAAAbW9kZWxvcy9hZGljaW9uYWxlc19lMi5weU2QwUrDQBCG73mKIZc0EEKSGquBnmo9aQUFr2GbTGRhdzbObIpFfBifxRdzm4B4nfn+f/5/4jjeOTsatEjewVEfjXZv/PM96M7BvmrAMwpY16NxkoFoghHFCeCHR6bLLChZ9aqsbvI4jiNtR8cevLY2eny62z+8wBY+I0g6RycKstZrOicNrJLd4ZBkSVUVZV6s66s6SbPAaepw9NpRe1r/w8q6rPKiqK83Cyb4PmFAuepbmbln7CbmEAcX1zovyttNdcG/oh4HOE7a9CtSFjNqO6NEULbrtIlAD3AZAzkPoeKSu2GlBeFVmQn3zI5naRrN3XKjzsiSC/p2mAT7VnlPq3tl5IIw+olpITtG5bGdn7hcH8M6mBP221mQ0WT/Av1FS6NfUEsDBBQAAAAIABN7a10FLDPFMgMAAD8HAAAYAAAAbW9kZWxvcy9jYW5kaWRhdG9zX3YyLnB5rVTdrptGEL7nKUa+ASSCDI4d6xxRqTlNqyP5WGmd5CaK0BwYjreFXbK7OJy8TR6hz5AX6yz+CbGTXLRFsoGdnZlvvu9jJ5PJDcpSlGiVAavxI97XZGJYKxBSFAKBpNUksRH8oEAKaFQpKlEg1JzCL+Tu/A+tqJVVsbdCuFmvoVDSkN7xPgTU7zthqbCdRjCqsg32vCe/XafPOBeeo9lyKwtYxxCk02QZeo+An/9GDrZK1UI+AJbYWrRip8AICS3qz58asprb8zM3KYjjGnaMFCUakG4z35N02fMv9iaTiVdp1YBVutiCaFqlLUg5WtwJI5Q8hobxjHez+nmzebHJ3vq2a5TOpZJ3wiorCj/y68em3ari0RK/cOTV7crFf88NQ2uw9t95XlGjMbB2eOg3LcqXPFIgZXynyq6m8MoDvhjdbNrPpvDkJ5j38yu4r9X7jgws+gWUwvzZsQLm2sXcFlHynKxcGbuxXIGSKqiU/oC6DAzVVQT9obS7RAV9zDS39PZJevUOsgyC2TSC2XS0yV2aWCbJtMRVJwvLdLAquHvInRBpGfQR/EVaUp0b8ZGyRQQ8KaPJFuEPm80jmH+7VX9a1CgMwRusO3qhtdKBvyEg05JGCQ22rOaeo8D5kh3hxA1BDaSc1tJnYeyHR9o3J79dUO4Yy3N2us3zPWUyH5LIZLNoMFw+uIiyV7qjEXrTMaYgjE/ZX2Z3deKKkEfjMtxyQywja4WD5jdK7pjFWTRbRE/DiFf+oNXrYHh6jrbYrpVu3IbFsHSH/cs97+nwfkzn5GU0+0H+0+VF/hnGYT7S2ZkvQ6feeHbgb4CcH24Hy9nH82EHyvhQ4FoX4/5ao7Uk9/hWQhLqIEmn02iepN9Dn5TBMfqLVq3qbBDPxwU4+p/zT0ofaLn4evjjOfjzbMhgzF7wldxBH3I5z9W670RdBnxy0thUBwsxwy6SZf7hNCykzO+lf+w49uwJ51kmi2FIkk3SxP9izCbbH1rxKBx8IPGwtSZbK0nhdXMm2IGU8XIsZH6cacTU9QFecwaFd3Gn+fRbOI6xCxBV8VXzqvhXTZPl95smy/+p6cWx5CCE3j9QSwMEFAAAAAgAE3trXYyZILIlFAAAljMAABIAAABtb2RlbG9zL2NvcmVfdjIucHmNWutu20qS/p+n4NHiLMmYpkVZciw5NOBJnJkAyTlBciazu4ZA0GRL4glvw6ZkK4aBeYfdF5if83seIW8yT7JfdTfJpiRnYiAR2Zfq6uq6fFXNwWDwvqiLyoiZsQg54/QQMx5WVZGmhbEZucanAg8Vi5JbZrC8rlgeZgkeCmNrbMI0icMo+fbP3B0MBs+SrCyq2girZRlWnDlGxDeOsQr5Kk1uHeN3XuSOkYX1yjEK7hhlGtaLosocowrzuMAv36K5TjJM/ZqUiyRlzxZVkRkl5oCEoRb4gFfZEYFPFtVJkfOm81WxzmtWNczk66zcGiE38rJpwo6jVe9lk3CQaJo+vH0nqYu+hm6ea43uuk5S7sZhHTb9r/HMWe2Ih3dFGIMHMQHkmjFvs3CpthRhyxBeXfBgM2r6b9dJGjvGq3dXnz5df3r27Pq/Ply/+u36teEb5tnoNIwXp2eT0fDFxAvjODodjdl4HMXe7fk0nk7Y+MX5eMQmp1EcjqfjqTeOhqPxeXh+y87P4rH57NmzmC0MHm5YQEdhleIkaA/27JmBvzorsRI1u3dJvQr4erFI7sU4Vz4bR4bpYphpNxPcuyqpWVCz+9oiqm4MgXOLqDpGksdQFX/kQHX4umJByKMk8d+EKWlHCB27C/Iwlw12R7Ni0I2IiZVtyTZUKRBLKbar4o4rtpOFkRe1aJlBVet1lYt22oPcTVGy3DLvTMfI2V2a5Mw3TZuUYiEpiNHYOhZxXydR/RdaqLIWjrFIWBpD5Rn304TXFi1yM5zb9oVxJ7e+YnTUVtdAQ8Q4xTmPiopxawuFVwxHGdbKS/crxnLLOnWMUxsHUW9L5id5LeWAfhyyG9ZWBMuw6JUMM9xaWwzWXksb754SXikpx0m4xDwwxdelUC1sLsMhZpaHxrJiMbbJ4q55KAmUZOpkDA2dTRIzqy6dbg6Md137LfvE+t2KVczvqF4aihyo4ZT3aCmmnqTUMN3SWXg9GqPnLZ/P5RJOx/lR0/I0m72BOrekOwZtwsrCqCqChecv0gJnsPDcjIW5RbIOo2hdhdFWddWlEKB9koX3lpKmOBHbaZWr+bsN0zCPWBzs0JCcdEtERb5YE48+CNaFUL0D5KKUvLZ/IzgWL36kSUIRb99vkjloy7V6C4uOPeq9P00Skgwv6IiEwlrqvKjDNuDSjcSJYPyw+nXGqhBGq1yaPVdGUcFqAnISsJSiVnZRhWSEouXEzMI8gfXxunBhlqbtiim32xq2ZDd2r2KLy1fhaHJmgYDtrth9nCwZScz4yTcaH9pZehUmnBmfw3TNrhHpKsu8To1uOTgTyD/JI2gaHYTBUuEk4dyNcB0ndRgXFzQIRJI8iZKwHxhd5RzJBWA74uwax/JROgvw6cYsKqDKtsvLNKnJKWFfygcuq2Jd0uQHPruphEQrkqcgiW1XNyYibI21i9yc+z43EE2oFfyDO2oysYF0jQhjzncOlqhxomaZNUSRm44pAzmFUdN+bGR7k8JpSk5u+Nzu5sm2OQn3xhtPTqfOaDiezr8n4FcFYnIBfAFhQGEKgTRS/DQC5Y3UaJUvbCv5KwEuIFIGx23WnNMPTjook3uWMm7a3ZLg96G6wcR5Jy3Fu9rl/NH4z6fHaBKYP856EtvfDTBRWDYwSJy90R4HQwwyj7CI3ZPjgzgytZv99fn88QkBn586I+/HZEtcyBU0QetyBbiqtFVdBKsMStcRl6Zb9U23quy+QKQCCneDrVDoxXhl4LNDDNJIgxS8BEVopN0jV7Y2D6ohwFWygQaTvfMi3TDL3l2cVizdhAcVA4JMAGfqQhDoptiH+Pi4BlpbrMEDZNXa9A43AjQInCZRQylwQpLN9twjOEkyN4MR/+SbH//4B9Mg6cH/J1/RYnmjcwf/DnLyBqgX0A8yATaAC64LUhpeVxTM9xaS3gxS2vF2GgZIMhthQjnHngvcFZ4iBtWCtHVjmh9iVAgiN7BJOMcIZ/cUnySMn0fD4dD3hzNEIIoK5mdWiWkA96bDncQxT0yHrAEq5SzSNV/5v1Vr1gu+UjWfPSP14sarqgCYVNBaqSGFjyCA562DwOIsXRAYLGoJCRGd18uMQKfElDODRrhigHrCIPHUDBR9olm1aKuAWbWI3UBLgzbQkrK1wUtWk0W1XCWa3VR+OwVRsgOdO9rWMtu3BaWCxr2/c+ZRUW61M8Yp6Fvr66yYXBX1dGjdNymXSz/4t2TWeP9EtTEFMM1Ld0Is3N/MnNns2Ot2UfsyKaLEJhAJl3WvWLNd2Hy2hhsZOUPHs10JIuyT0WTSzifYo0jUSBSgeDdD9+x0Ohqejl+MzkYT7/x87Azd8WTqvRgi4Eyno/Ppi/MztJ2NXrwYT7zp2elk5I3PpnMb7P0CPyz+63jk8d4KI+90+mJ8OvEm51NvOjwHtRFoTM/OpqPR6XA4BVG0eS/GWO309MX0bIrUa/TUCko9rPqY9mOf8C6TcykRurc6r9mkBoygYJpa9NCCICF0apHNFwR8D7TK7QC3rMM06DqasCO7o3UckqsMN2GShrcpfONM79Kmd4wo5rZ5tKqKHO7MitkmiViXb8l3V2QsvkmUzB5Zfa7kiKV705AntrPwvDNJ8MAQl9cUhMjNAtKnIq92jB4/os+loZbIbYpbQOL5hbGVP4AZ3B+6EKM/7PJCuWxeBEuEpN0YeO8IACJX61tRWiyTmvtiTWg5Io/iBUsDVee5u1jnoiIRpi5SCM4DQghkDHKqs9UnHYptkrWEL8jHMStND8aQD9/+UVEFgeK+VoghYCom7oVZkouLNJ3lsWLF5cWipqwFdhmVa8tuUw37YtuO3XatO5Lg/EglBeDxOfnFrX2RH/nySQwOdY8lWGgVlLaal90+Q9slDbQpjKo+vEdYBn0qdXU8B3Ez9T12PDksFSwR3iYkjpgRmNp8+zu9cLMXZYj3k7yXRm85pXZS7aTnZRYyyKY0ElZw7z4aXEA9nOgJdbpUGTgyg4/Xn/787rer179+cr8mpUnGmZW+nNMrpejFE6GGqszl/k9SvsGvhW6HChVt+9sPwevrN++ukMGIMPC1r6kl6SmnRCwWDFXLtLi1zOemvQ/aBGYiqpAx5QtlU9QRWNul6g1yAZFt4WdVZyn91ve1ae+jn6+uqsY4VK7pcBiJbL+QI0XROwLZpFzNOsvCaqvJG0huncLObkRNiQyRW6VMAUWpSeWZu9tXuz+R05FXyF3Z9lydIjwgnEKLiOn85PaVKUR5HtySGGKECZaz2ht5eAM9PE+GuiSqyu/lZpJjhY6FfxCJGC0x1yO0qFVVM0otAdHXrO0jX4dsnnRSaORN1eQlMs8DVVWU0DOIqprbepSTBilpubyOrTguFr4HgWFtskzZZV968MmwHgpiXSQnAblhWZLhyzqI2IjYhcN+Z9FaJjl+A+MaloKMxUnYX70pZ3ynstBOB+cImNEG/58cIiKwTWwkXMiPmBY63Bt0Oey2pJSwKxtCPU7oHJHW5IHcFZe6LjYth98W8dY3X668yzeUs/TL4b+sX737ZGxGL0/Q/7K8/Kx5XaoHJ18BkCkTDMEGWyaV8e0fZRGFrvGxUUeZ94ai2k7VUYGpkWvBAkTNJ1lSVYgoXtAw3a8jgC7FXEX05Ul5abZMH4Hr8vIdHARnWZICZRhZAh0GiSppHCIt3itVyDJGkWuJIyEVOn35qq2yq+idIawQHIpq62umKqRtaXYAJykTB8o6JIdAQfaJKScnoTLUnom3K5QF8gnumwa8FB6thfkwHh4lzyfnQ1F1I2VUXNjHHmDmzPUWj87DyBsee+fD56ubAUQZNMo2mIt+U+W7q36+2xDqlpcCXkAvRpdI5AdyU4P5o/Gvv/1fI3CqLQzUM7qgJCNSkvfsdywiVj5+483EqM6oB3NQ07gaLx4v1AGLkRnNDhi9g6ZrvFd0IJFGuwxr+K+//S+23B1Vj2e+WRqbhN39obj3B0NjaJydDY3ReDhAAIrrlT/A+wCchohGsT94P0bv8DMk96ezUwxCvEj9QQ6LGhg4v+IL8weATNvByeVLOiYkBIPJwNj6A4y+9F6eUONuF6gNLodNX1mkW6p3Ncc6eJAPj4dXq1mYNm/HiudTWv4EW1M7ltb99pc3v358fy1Dl63fTJgvf4qLiHCnQZ2XLzNWh0a0omuq2h+s68XxOYTA623KLkl0Dwt45+MF7CTdzq6gn+kFFE0uP5sOh+U93qtlks/GeEbSCWOiPR2vWLJc1TPPPXsEew/dJG84/BlaIZcwj2iRXjzUMYeqka7zAJKwhO8VuYKj0mOH7tj4DgzGcJ+aXS7U68RS4ETanQD29gUGudmXOKksdg89D4ovWgoOF2thgHSTevh0xeBeqUil+P8NKRZZmTIaPTMdwatgVcvvL7RLmX6mcSFMyRf3XoJfWwPHYnxR1kmGjKBS6Zt4d6/iMPuLjE2ExzCxZhX4c9JKyiCtnDtxEkHMonArG/UWVWhlTeL5RwR7mD8wpP1ETlVXfne1Z8nKhCArbvGcnWqjI3bu3IZ1BPSHDUgWxLvDVwBeQCI0xlk2K/t4cuCGgrui+oLt+OpaYhP+0MJ6CfOphQ9Qb7w3UqVbxmv/2HPFg/A6K394cRvG+J8DrzVtngytKxZ9EZbrC6WBxiRZ4Za12ShTN+KAAv2HvFruxnBKUcpExkgqZBtRWJWM7hFzpFShshDZj+OO6F7HbelFX9RJUgSyOrJAKGWQwlGSZJCrlmtTqQaHdaVbVSjqihFCqYhIAB7gQOTlyhcVyyDbi1YnDw5re00NmEnW6O5VjBECVvQcIT3h6Ul4tiYhUnPCBZxVm1BhBDK0iiDDqw9/pus9BPqCij1CLrdoJ9FVxu9rKk6HKeGRb3+HfVRFJytRDCauKfJ1uxFNroRU1g7m79Xl2X0ZhJslgLF6Cvhf8QJnF2gNB7IGKIUiItaayR9Rk/e7x10foKspSU89m0pjG4Gqd6WkrZjFK/VBj0VjGONN12dqVaOOPK3SrkouteBMnFu5rVdFHlT5kvRAr8vUQTdKlMCaQerkMQAN2iDRrga1a8ID7NCSNYjdcU+UYPQKjL6gcLhETU7apfZUbUYrzeyxf5iSigofAcfBhKj7dhHBlIDFdDTZ75WBSc9Ej4CaojKpDxe+TOAhfuT1739wrpfS10k0SxdzlCKIiZeiI0tyCab4zLgF0vzSJU77xa6LOol8+hiFSpiLIJLflVgqZrnCz+NN/AZNkSmHq1Q1fVVt6uyNlY5172ztPtysK8fbsZTOIunuWlao6ATqIiBopILqdypRqlqxPWRI9Ce4/YFSlZ62a5I+VKTCbHv2/SpV//Ohp+pURAnRKvpyF1Yx5JsXVUa8yi9uojQphUACag8OQYCJ6zBaPEgWJC7JoKZh+1Kmo+mdZFvWIl5ihKBohXTkcIWr+VOnfiQuxJsCG+Qn62u+qgbZ8vOAPZESCz978vpEGtEBu5EmAJdAN6JCn5oLlfrAhQr9bVJn61DZzd+ppG5Cp1FzCM5vv1GhsR3X9sUBu+hshkX8kIEcw3A6L1rc+SI0Crvz5RY0m+keT3LV3n4boUSKDsrexHBsaL+aoOd2PrajF0rE1Jai6Gze0MnZcg0vxX3aywHK8gOLQH7Zsgpl2QNuf50nf10za0dathAfnGTcq4aIUVSJGdpNNWMvrjU1F4ir5+V3dnNJEa3vLEQI3Bmmx0HxvwRxvXlUodSQB9UlLzSUIg1Lgza2gyEwEq2q9xRwoSLMDCse+d5OZEDiTnr92CTN/gNp+KNKZf0Hweyj9E8PumqI1Fgc9BvPf6Dt7ubNRp2wrCyIYsRFfs/NgzZBBbriTl3FaGdLxTpPGZ959fntp19n6qMnZqxzwK8CSb6YdEGfflRsiaOrVMGnYvQN4bd/Uk1ZfBlSRJRBuYdZEGL1u/qaf0DaXeKzA9CaAdrxOg06Ur8HK24d5tSBcv8M/y1YllnNU+R9+s/RlK97dEgD8c/pIJSvwNOyQVi20yInv8NWyxYPHV64BVJqW8seVkFK2CIoSrD+LbkO2PganFruwSn7CRAmS5AddNphZ38WQS2tbtmlDI2BqsRq10IFwX2j7DIfTd/abz4FwX7NrVEe+6Irloph8JKbUNVImzEtyaZcoVUcbO02bjcx+kGNk5I5oHIqE/6BcEYm6/cdsyqzIJzEzcdyTcHT732X46gU09dv3x1h8wHgYuqr+9wb8hFb+txtfkiH5ITGsfQmUaOcVwb1OkPS38WIm8QZInOHOJJ8UURJXfQ7PdHJqBKWhf2u0Xz3u5GDn/IcLBXMdwrl4nwk87LWH3T1SqkMSpTqi1HQgfQC+rRBv1pR7doFzVPfQClyjzqcv/dvkic+R1KLt9/BNQR9Xy05v8haQHOT3EPUQuj3ep6iMd277mj1Qo1wchHyk3vtpgMolgfq08ushzU6nNFDGSrSkAapmxOgecmbbe9dVCj7qKsEeUpQAsg2TEnp67zLuTykCpyfU6VTfLi8CW37Zqjn0RcUQvKluBHX78t/8EI86LKxiT2TKYdcVXMy34GKOzRGw52M57upVrPSITDa7Ks5ROsJPGo/B7gengjZS2qNV6BKp7931dXgEwHAtSq8HtFkJhmIW7E6jBU6bDzlvmNQWUpFcBNJQOmSTlMgaW8y97MZW4OomCNuT+S7bkFPr9mZro7xWWz3IPUPV00deCZWidw6yKRyCnuEmxeIN2/8EsK3bLDUAWEn2lxRjfS103Dkl6u++ebq03Xw6tf3H95d/3b1+ir4fPXu7eurV29//cVslL0Xy3ZK1Y58pzAgcNw1fS9OS86QMalbZj1k7cEz+jJQxpQWdiEtgbQOf/qyUycBAq23QYR88TvfvezVavqz/h9QSwMEFAAAAAgAE3trXSFp30s7AQAA4wEAABgAAABwcm90b2NvbG8vc2VsZWNjaW9uLmpzb25tkM9OwzAMxu88RdTztGpDXOA0tlzQaKfuzwWhKGsMitTGIU4qKsSLceXFcFsQHDhZ/vzL5y9+uxAi6yCQRZddi+xQyb0qjvJU7tVpkc2GcYsGGiQeP3DLQo2uc/AaVbSuHxkWravBR7ZR3eWPRvCSgPWwNIoy1h5HQ9KNNXpYl7NVBBdzE2wH+X2/GWuR1tu9kkv1X5w6aQPB4fD+FwxAqkjQITE4t7535wkHitqM8K6Su1W12pSqKJW8k+vjYWhkodbldnU74ewDoRvDMbWVspp0DPYZnGoxYhiTe06qI+Sx9blLdUMKlmq4ysQ7jKPJYilqDIH/S8KAOH9+EN/EaNGLxdWgeNuw5813FehrvqFuBIHXgZPPhEO2cE82tJqXW5yLAgUlijam/q+lD9hZmp7rOibdzLOL9y9QSwMEFAAAAAgAE3trXRufjbPkDwAASyQAABkAAABzY3JpcHRzL2FkaWNpb25hbGVzX2UyLnB5nVpLc9w4kr7Xr4DdB5JumnrY8k5XBSdCLZV7tStbGj08u6tQMFAkSgWLrwZIWSWHf8we5zjRt7nqj+2XAMhiyeWxZw+SSDwTmR8yv0zq+fPnv+bV761g4r7OK8WbSsmKTXcjdirUXObs5Z/Za5ZW5VzetIqnsiqF3iqqTOQV9dUyr5qKZYLtMS0KmedcR6P3FeMzJZgSWqg7HjE0YI6cy5Sz6Y55r6mzbDgTebdKWhUVm4kyXRRc3dpdVWFlikbPnz8fyaKuVMO4uqm50iJccL3I5Sz8qKsyrHRY57yZV6oI9VKHjSxE+CBrHEOM5qoqWM0bGs7cMqd4HVU6EuWdVFUZadFkYs7bvPG9g8tfj/fPk7+enP3n+en+wTQ5OHn/9ug3L/TGr7d/eTP+kxeMsElES0ayxDkbfzvUjfJpVT9JaNckCSIcs8rvhB9gqMKB9dXO9ZZnNai9IBj9dnZ0GF999nLljXMVep+EvFk0SSZSvvTGn7IvDCdiuWKyZP6OePk6xK9XgWn9lJnW7YjaXgfXo/Pp9PA8zqVufMXLG+G/2dneCfHrDXbC4Ri0WcjUJ7HDjDc8GI9YU9SxOccn2SwS3c7n8t6MiOzzz16EIV4wwe/ok5KNSBpx3/ik9Shri1r7tFQoywwHjHdDUepWiYTrVMr4Lc9hKZ7n1aek5KV9D+xiSsBiqTCbWflgpwdRrssn58ZykbjHubRPTdRmts8rnmkrrBI8s4IFwbOYJo8Vl1qwDzxvxVSpSsGuKyg//lGyDCtKoHDMvJ/JeEaQYARQajH+SldGwkzeCKi3DsZKNK0qmQNhpBd8d++NX1tBZstGQNYgWoh7N8XOr3l666uqaozqAWXRxPTq8LFl+qKSF+JnLzmbnl8eX+wfnpxHQLI1QWwnrVnLWWjEqJE50Ef/I+u3+OujD7jywr796DQ5nL493r+YHgaMa/ZAKiVA1YQnjcshMiuHusmrme+9AFJpjLFFJLWBt4+5ZYZ3K4SBohel+g63JCLr0N/mvqE/i6bIvW48He5Z7E3/Y3pweXB08j7ZP7g4+rBv54wfLMb8OiRV4kbLOwCusjoLDFx74FhVBBNnCvs6Mnq+47mE1UQCt1LnAguqtgxp61ALkdFx6hhNWx6uKO48zyorgAFcWTUQtEecW99gd8RUPMTeGvDMZF9duRvuXYd4dp7RvgAJ2Mq7Bkb9XprQe7t/Tk7m3enxFPbeTz7sHx8d7pN2vGADjs86maF1OEoNOQEe0eEY5yJZyKRza5dCfKxUVJMxMFEWlX1eYCL8K+/slbZw2NoZET46k6l1+onVJ7243kI0Cg5dJ3CmCVAtSQDT191QUiKJsjUPBpiBTMMOqAOKw59Eywex4ahQOoWJOwnvkko+OCKm45CLoTVM85NTBV9bKIePWcACZI+6okOIjyJtoVCuvWsSkc+0X/B7//7Kw8mTgqeqSuY76CSlGrAvgpfK9Dq9eNcwez8u+DN88i8bzvPvTrh1y9HldSi7UK2wGC64LH2L1C7mRfvqpi0w45TelB9M6ohnWcJds++9fAnYI7aEzbIWMYWjUInfWwljxrT0phnaHOLbUyDBV3PmXAsvTBeVTIWOr7zaMAYAY9ZqUApazrORHcB3gTXuRm1ckbekmAeuvNV4GsnjOjKHp9G4jsaEPCIBnvUrGt/Co36NzsFc7J8lF/CjyfvL6YeT8+TDjrfBKNOygfflBYEYJMSQItyuMWxyJzWHZ7R8aMn6DYgzwXXwwuDe66Syqux83Pv24Pg8me7+gAiXmrOcs5SrWhDg79O81fKOs2+t8WRH3DC9LHJZ3vqbPMa5GcVEmfMHwBx3kM5UyIYMT0p268T9gj11mfDIQCp2f1c9axLEXb+5P66RLko3zTEg221Hml63n+veIPshjd1yC6IPu/Oal0Oxo+I2kxTAzRoGtaFx30l122OYjFnFTzbcWhlp//CIPO7+MfQ8/a/T45Oz/YuTs6MTb+sbwcqSE7PsIFgY+c/akhioO8ExZ7PHf9h7AaQ2ApxWsQZwE6wtOW66AjfmKYKd5ctYQZYS/k49/i+rGgXLkYeytKWU1A93IZXp162DbGVw+BMtgPBYIx6ib+YgpXvja0acvyZhSkZOXskbCUDrFJEX5Egzw9Otdculf78GrpUHXEFlQBO+CT14OyMB3AXTspg9/pHLlEjwiOVVetvb5dvKbtSSQoshOTQlqmo4cu/eMyxmPu4Zqf8ZrifzxgAdCAEe/QCxrtKNN+5yhKhEhLbNintjslVEv6hpBqqR0HxD5r2tWlXpFuj+1q1Qpci3wK2zqtjqhq0FGEQyJWtoCbr70ekddAzvhPFK8SWcEw29JyMyInFTM8QodCPA/puDiC6BpR4JBIKcl6Rtg5+0qiWfsDtAzyRiWpK7GaCqh6UGQxYdGMk8pHdxTwFS+Jx4hSxB6JdjY4O2NLAopNayvFldNxPBVrOIyNu8q4SBlmSwsg7hS9OF/Q0/iyBKaVvRD00r5BF3uzSYHkfM5HEcrARDeQ5aIna7se9ODqfH5+GslXnmBpp1o7aRuSZ/w7uh8Cf8GHxBqJ7s2aFpax0pv+My5zPiKxu1PUXQZmlLK8CPmlgpkPD+dno5Icdqby7APgwqomQHp5ddoKBjRkkCa9CpkwSxYifajnZ3vY0bnovVLjQ1jt1oso49JcJjBpoLl15SVpMmPL9BpGoWhfZt3LcDZ0hARJlpOmxZRn3CHTt2u3HU2srGwBtWgxPmTdHmkc32mvmrXbvo5p2fjhqxDI40FbEdbV98jxb2DOhy4hnr+XVoXgkb0ZO2H0zBt1JcRmmCEYAW1UvvX5u+jkWaf02BBqwnrfIYnY3fyLqKPcvt303fX5zYQG7werIeaJL3J7bScPbORZ7QbWSTeovxIOxqLYklWLHJ+8MbRY0xVRTCGW/SRfwKybjht/HuXmg5OoAZ74UwZeJ60IFIAtCVYHs2W0dDam5j7M3Bq5tXu15o1NDSWeOd3T+FBS/lHLlMbHSPM0wPkE2GlicQP3cRPsABYM/489xQorHLhefBKjkxhv0SumoRONNJcjydnnk4gxbwLSKmFK0Uzd42EUNkA3G0vRuMumLBKm50egcDlx/7hC7smsmjlnfOKIQxh7TBNRy6onjwvD6GLuDTCxwatxaX9VpjvWwWWKkPOva9G4BYc1O38cD3IFolFvgJqcvfDjadk/wJHM/6KXE0jL1RVVtraxUTlcjv9cYYGTcaGR8ZO0/5E5veSWIKlCgrWYAINBUSS5GzLo/KEQHIBRNvfPwbwWaC+IuGhUhv60oSqeOP/yiJldy0XIFWRG4r8JoFEvg458UMs6u2GePHdeq2gOdZftVJkQOJnD+vcnjYVdIeLurAliqo/fuUb5IRQu3oLd/VVGz6aGoAgc1Tvy4V0LxhrQB6oRTl5N3RBbJy1uXp3kC0Odj6wm3rErlhXkorflVqeJqZUmVTr1K991hc11TncATb6WONjP9LN/3FC2gQ+wwQR9euSWrBb5NCFJVaJpSRg4xMLIbaMgEfGZQqLMJCEjW0UDWSa6qBfl+VNNTW1axGzJ59/eEzEuj7hAJDikVcFQ3kbCUu9Ts5+2FE2ajdepDvTetG+cEXksaZ9ow0nSPj80JToesuWzA0LIb/f00LH0R+bkBybUY5yCNsRhv3Ge3YbEerIuxdT+giw1T2ah+oqtbdtXYmufKMF0e2bWusmP4Tu0RyQekSaxDnONbjecjA73CnQUjooutJl9yCuXDiR0hHHv9mizsMSKSUKxemDgQv4eIR7je7D+Hv6IAS5MBfkSrfiWqhaeo6hE+9aOdzqMBVgIPJfXwfNZXvQDRZxsvBq6tIEnIoSNjIZ2qQ1nUATIQA/9+2t7eDiZHKejQD1WC4cFU3zsfiSRbRfsaLv/pmCsV0DIf8QHyYK0SW7R3a+8euCDE0YxxoRJsnmkpySxLa1uB3dm3tdLikXoJ1qaqEaswyaWwSEDIErk5LTNx3p4qMVX1zjOhBqCq5UTzzSaSmSkpczh6cDF75RgIuZqJ/Hxi53NnBuOZtCVJPbCVKFXoSYqhVvfTttHBpF6ECmWmgmnYtnsU+rPeqr9zZ5aRGFkB1Wtpic9ZXAucZm8NKeKC6nbGMk1Nrwwc/IVr4HTWFiJasp7mszTFxPlUkm2y1FwraJ5Fz0oGVxblfUhQIeu0H31M7rBfxGklk5m/Q/0vYJXCm7YYZPmTOTJyYpwvcbrONFmlsO0EBCuTUvDRr6qvd8XVAMYFusiUf9g7FxivOkR/k8NQW4FfUdn21fR26wyqiU23h1xH4BWwarOr0XyuFiOFNi+RSJ6CFIjHIiSFZCBclC1OlTPohJiokmuhSP/DF6z2EDOuTQS7TyvrS+PsemFSS2MQnj63O6Bim2eSLXePLnesQMALR29mFe9GVHhvqwRs5k4SUzORQmShaSE3ep5rBbX1sKefKqFzC6HBUBEHgz6RLr8weJrvqvWZnM/saTLoPOtYVh25UMLEhYPA5y0144vqxeW6VHgJhk4FKRFE3yyQFGkxNjFltA49kOVvz/672XWXZfOfrJFsFyyHHRWdNqumIn4EUVTH0YDl43WRnu4v8nUAvdrZf/LL16s32dvjVBNh1d281cnevG2mN9ZpZE2j28+BjL0O8cCEhYvuPf0c6pFhXEn/8owzZ0dYJWzLwX0NITFVClmneLsXgW68xXshUNaMtHsi2yL7tEpEXDOL06fTs7dFxx8BO/nm03hRUbQBffeSxYFiVYOhjK8j28AuPGzIM6MGzmEa5fGxzjcB+USDBC3jtLsgCwUSdG5ixj6WEWhNjyTkPKEIfd5OFxPSVS9jEGvqyOwWbriz/+cu342hDzH4tZoFf0xhBrkbBYj6lki54KfpYNsBhv93W3LP/IJB8ll8cI0ZQ3iGyPrF79Bfxn3wuoU2QipAfEGOnND781wMqUYGygMrYPJf5OaAnECCAr0+ZfQwi85Uypk83NvqSmawUcI+3ok82Po5t69XHa1Bdqy7rfa2PXvunh1iCOpMyruR1iCOAgcAn2lVJwTZB603YfRUZ4PbXy/O/XE7XcocfRq79iDI06wCd/aZPcwksAY7gd6I8i+ltHbBr4dpijbRMH6YMezd1F1jl3N72Dsrkr2dg+y1VpX8H55IzxV1dm9xzSh+QCehw9+YGOwwSZyN8mWrFuEPdBmSyRR1/vh2vWeXq1jrIW/s9M1fek/+VCL5M1hHq9LaePm5yqfaT7NCb2pbYOz06Prk4SfZWX2aHFZmVAwhxZIW11hpf0LM5a9AVNxIB8LTQVVdooQBG349i7xCqLbXl2GemVl5X1mdOEA/mQplnUzmHaTScpahzUHTnZjvXCXJP5IVKOtyFU/po9Q1nao73Y84UcEpMTSJJgMkkoW+TSeKN7TfK0f8BUEsDBBQAAAAIABN7a10vmwiafAMAAJcGAAAZAAAAc2NyaXB0cy9wcmVwYXJhcl9kYXRvcy5weYVVwW7jNhC9+ytYAwUlVMvYjux1bPBQNCnQQ4tgsSiKLgJhIo5iYmlRJSnHyd/0A/a0n5Af65CykzpZtBcDJmfevOF7MxqPx1f74AAZ7mvTe72DLbYBGZ3p9mwHRisI2rYMmNK+tizgtrMOzJq1lpJSrkOPbgdiPB6PdLwODNxdB85jUftdsQG/Mfq28Js+aFM86q7RBkeNs1vWQYh37JB3TX9HV39cX/308epS8sXsHFRzvpjPJu/nU1CqPp+VWJa1mt4uL9TFHMv3y3KG8/NaQXlRXkzLejIrl7C8xeVClXyksGFb6iXLVyPWySMv8aO762Or1/Gfy/J1Jwi+gsNxxt+9I568CA8dysiqcPhXrx0q+dH1+K14n17rP1JAdiJVj2k+y0fMWRskiCFT0Dtas0MiE8/F9rPSLqMEwvcJosC99qGynwfAETO2BiNT9BB3xtu+Nr7CWUXCWV81fdRTxF5GTDekWhiyRMLy6V1Y2HZyOL3XYVP5vmn0PuOitp2Gim55vh7Ui0cPUb4MImhBd/mafoh8Z6DGLMEQtQjEDlKLP3X3c8wZLhl49hjLOriXj5QJKuNbaHWj0QcryDOcECLdg3WE38BsvsgoIRcb3Ct9R5FZ/p08emVFhvXIfgfT45Vz1mX812fE6N2gW4Jmv1nm8WDcllz39PcdtuhFKgjGOHvvpaHwjFiIS12HD0SPHEKlhcLaKtJH+M7oYDQlZnke5aKkT4411jHHdHsEih24T5ykCbqmKeI38TLjabp4wV/mi9OjtCoGE19Qlt9IyXVLMxktdRO5efIpaYdtFqFzKaeL+XJJN7GqLlJdbMmNDgKmmGKaxGXKB5lFk5wRPrh6o3dUIH/xWww6eINihfaVQ0O8dlgFmzLzb7zvhz4AI3852g3Ikt34EeoAk4wytPZKyRgQha9uH0J6xX/LKiURpcDqmeyqti0p2GPqBwIcfXPS0aH461IU/so1b+DfdvfLFsgZR+fAivEf3tSKTQxz9//DGodExoTTCRtmKw7QvdMBD8+RKJ+MFWUe29PfzyaTiZSTVed0XD3XFAQOav30tU1SoAdqwhi74oUu+Bkvnm1TNLTmN8f9wQZbvBm+EzJx6Gh3DrUuo8wsDoj1KzZdCPLgyxidVl8fvwzP34qnL7TmyCTURVW19K2pKvJ5VcUNXVV8NWzq0T9QSwECFAMUAAAACAATe2tdNrir7SUBAACfAQAAGQAAAAAAAAAAAAAAgAEAAAAAbW9kZWxvcy9hZGljaW9uYWxlc19lMi5weVBLAQIUAxQAAAAIABN7a10FLDPFMgMAAD8HAAAYAAAAAAAAAAAAAACAAVwBAABtb2RlbG9zL2NhbmRpZGF0b3NfdjIucHlQSwECFAMUAAAACAATe2tdjJkgsiUUAACWMwAAEgAAAAAAAAAAAAAAgAHEBAAAbW9kZWxvcy9jb3JlX3YyLnB5UEsBAhQDFAAAAAgAE3trXSFp30s7AQAA4wEAABgAAAAAAAAAAAAAAIABGRkAAHByb3RvY29sby9zZWxlY2Npb24uanNvblBLAQIUAxQAAAAIABN7a10bn42z5A8AAEskAAAZAAAAAAAAAAAAAACAAYoaAABzY3JpcHRzL2FkaWNpb25hbGVzX2UyLnB5UEsBAhQDFAAAAAgAE3trXS+bCJp8AwAAlwYAABkAAAAAAAAAAAAAAIABpSoAAHNjcmlwdHMvcHJlcGFyYXJfZGF0b3MucHlQSwUGAAAAAAYABgChAQAAWC4AAAAA')
assert hashlib.sha256(raw).hexdigest()=='37e916dcc735fe86eb2bfcc142b6ed1f98b1a468337fc07e03b4b9c6b291dd05', 'Paquete alterado'
CODIGO.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
 for name in z.namelist():
  target=(CODIGO/name).resolve()
  assert target.is_relative_to(CODIGO.resolve())
  target.parent.mkdir(parents=True,exist_ok=True)
  target.write_bytes(z.read(name))
print('Código separado verificado. Salida:',SALIDA)

CODIGO_NUEVOS=CODIGO; DATOS_NUEVOS=DATOS; SALIDA_NUEVOS=SALIDA
import csv, platform
import torch, torchvision, numpy as np
assert torch.cuda.is_available(), 'Selecciona GPU en Entorno de ejecución → Cambiar tipo.'
filas = list(csv.DictReader(INDICE.open()))
original = next(Path(r['ruta']).parents[1] for r in filas if 'NuCLS_V2_RESULTADOS_pool_corregido' in r['ruta'])
cfg = json.loads((original / 'configuracion.json').read_text())
actual = {'torch':torch.__version__, 'torchvision':torchvision.__version__, 'numpy':np.__version__}
print('Entorno actual:', actual)
for k, v in actual.items():
    assert v == cfg[k], f'Cambió {k}: actual {v}; anterior {cfg[k]}. No mezclar sin documentar.'
assert platform.python_version_tuple()[:2] == tuple(cfg['python'].split()[0].split('.')[:2]), 'Cambió la versión mayor/menor de Python.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'timm==1.0.22'], check=True)
import timm
assert timm.__version__ == '1.0.22', 'Reinicia la sesión si habías importado otra versión de timm.'
print('GPU:', torch.cuda.get_device_name(0), '| timm:', timm.__version__)
if not (DATOS / 'manifiesto.csv').is_file():
    fuente = Path('/content/drive/MyDrive/NuCLS_V2_DATOS.zip')
    assert fuente.is_file(), 'No aparece NuCLS_V2_DATOS.zip en este Drive.'
    subprocess.run([sys.executable, '-u', str(CODIGO/'scripts/preparar_datos.py'), '--zip', str(fuente), '--salida', str(DATOS)], check=True)
print('Preparación lista. El perfil verificará el manifiesto y las imágenes.')


In [ ]:
HELPERS=Path('/content/nucls_e2_cola_unica_v1')
payload=base64.b64decode('UEsDBBQAAAAIAJKlbF3mlaFywgsAANAYAAAgAAAAc2NyaXB0cy9ldmFsdWFjaW9uX2Rlc2Fycm9sbG8ucHmVWN1u2zgWvs9TaHsjqWUUy46TOF4tkMlkBwE6TdF0isUaBsFItM1Gf0tKjjNFn2Iv9in2ah+hL7bfISXbSdMBBkVciT+H5/f7DvXq1asP1V1rGvk783LRyDJVwnv0RKPVXZuqb/8rPbmp80qLptJKGC+TXlPVI0+WeDRC6yrPq6lXVp6WRuq1iF69enWgirrSjSf0shbaSGYeDftsqpKthFnl6o41qpDMrcJr1DYqP1joqvBq0dACr5PwHq+9tLItaqhmvLLuh6BUunIb31+/7TddF2Ipmf39u8obqQ8OMrnwCqHKIDw/8OqkVyy60Mu2kGXznt50EE7rSGQZF91w4B8eplWmlpXPmsdaJqQP0/JfrdIySz7qVr60pWzlujJ/aksGB//JHVqt5Z/aYUSuMvEHW0RSR9YvtM0E4YGHwEUUkkiV8E8TDJhpdCAi55Qjv6gymUPxMJx2zk8rLfl6SHGiRyYylaqqFLk0XNrhKs8guJZp8jQBIhrjFEy+ULnkeZWKBluD/hTe+1VE7ml7/tHTU6L60Q+npXx4fgKWt5Bsz6DTAvoJp/Qb5ZXIpI7kBkq4dQEkwAe6qhqTzERkPX7kv2sv397yqyG/+Pn68vrm3cXbK7z+4/3bmw8XH28+XN8c1Sqvmor0fL7l4wesfffb1aebW/4p7hfOD7xSFBKH+KiiUjbxmc9g2x3cgLf1iJtC5DmPBwOMy8UCxiKmmOJ3NJJJ1fBGlY8csUpX8QkfDo8xbh5UuTd+zPGeVQ+n3bQ7YI29m42hd7WRmpvt9rQq16XcONF4V2Uqa4oIX49IOpIHgCH1MOPGn091W5rky9cDb1Fpa4+nSmcXas56MbGunMVzTy12K+RD9OvNz1dvbz2ZG+ncPRtAnk5m8IwVZ6TMaLEW5VIGJ/EgZvg5oWr2LGZY0UcL/wuJ/cq/0Iav/lQnBDs2tiYIaOERubjNG5FVEc35YaSlyHgDQwOksdaRqGtZZm4502FI2sO2GYmeJ1ofEAQmBokls8AayO7lY5KL4i4TXnkeHJZ1VEhRBjM989e26ig7/fnML0SqK76I/bm1izNtzbLi5/OQlWE4Ox/BbBG5co2K+0xp6KIRcWMLlcmNMg2v7ruyXeqqrU1C5eZsAZgIFKnFFFhUPZjErem1aaw2U9IxsRgaAfphUDCLTkaT4WB0fDo8GY7js7NjFh2PJ/HpYDyaTCbDs8np2QmLToanp8fjeHIyGg/j45PJHEqzd1Up7c98arLnYofxaHJ6PBrH47NJPBmcsWiInZOTk8lwOBoMJhDFovj0GCeMRqeTk8ngdDx8JhalKFMF7PqC3Mtlan16jlhQLhYqz4XhPXPRzKzPk+F8u4Lrju/8czvr32kMV1gcnbE4GuLveE4VZWS5qJDg3KhlIWh+jDk2jDCrW5VV3GQ0ChnRALoPxnOqELBNrWXj4n3uX+3IsyLO7JOBiLU1AumCwApPfCadph6F3HHwUgPSEHH57T/4xYtgRLK/4Onw8uJXr1QeKjdzi1MBWTmVZKpV3XCzEsPxiX/esW3k3gOC/IA7cOVd2t89NhJIH0YruQGkS4MioBJO+vw78msNjEqrvKuXAyreRWRzEDs9UWbeXpEtnpTTXxIXsnMtFGr7k8hbeYWmQQf+JcoFfoAZGHQumZJPPHeut5baEKSDsVCBNrmNWEtOZwUL5uQSRYmizgk7kXZSlomRDTEXVZdy1SXROUgEQQZUChYzqF4DFGctLJJKf87wliKD8EgFDxuxBi5vSAIJtoaatrBCZp/n2+VJsttqj/1stzi1wr+enXePPa6o0CpK9BzgDJxG3UlXKYrdA6MTPwdzKVAI8FnmycAq/aCalWtuogqS+go/sgopKCRQcmoNPgmJZ1Vh0VEVYMGIsJwI3P/wy0/kT2chHZXsJbt/7lYvbNsU7LVQ0S/IMaNE+VPe6sBqZYERPV4CtBPUCorHQBUss+0FxhZIiGY0DI+G4/HB/nl9yWGr0K+trCfztry66TcQBNDPqiKClwSAm+tyGUxAhW9UGJWVBjOiLbFSmNCU6zYtgBVNq0svcEBkKd92kAEkprmqA6HZgMUhdCeNA7/TF71MVEtdtMiXoV1xSDgZHpnsKbUBeMjBKPe1qlqTEEY9o6s9BLKxgKebXWV9x1Y03eH9M4hPVzK9ryvgS7KjohltO6Qj5uBLNEPyc6WjuvGnq+RZ6e/2/7jsu6QISIsjkD8lbbMlyb7erR20bp9YX96yDwSokK0KPT7N/5KsXgKG7UIAGzACTO0ydudqhJBO7s7doqaJStv51VpRx3v16eLtbxeX19/+/c67+fX64/XPFz4jzzFyHFvkrVn1/q1KNDktxa+DGqzg6LlspoVT22cm1BBGd63KHe2Hf9TGoMvdXxpGaZuJAH4E3AWdQGsENw2giaOBbbpctabt3MUKUW+b4cQnOT57kGq5agyvyvzR2QCZorYwKBoiQdc8ORzcAyQXv03S400YIZ9Q+vJ3GQy2WnYXA8OJh3jgnJRXS4UWxKoebMJpg1uCbBLytZuKMFCITRCH1JSIvgkQbVPRa0Q/3dLZgLntc7YJqdtDEJOAFrzeoCTvkGkRgW1sJ8myHj6xEFAAhl1hSVq3+HV1jUMrjRY+oRUL3GZhIibpKoNWjXIlxX5VLruYGwq06dxk/bTQIrVlO4jQn4wZKD86Jdp31evhCmnuO6NQ7CaIh2ev8ccy5GUqu9hQIMz9zOoyOyfvkNzX3dpwPk8GdkVCP/C0RawgZvgHUbTEHWbh3h1WVjYQqD6ndO+LGUlmFriCLixb37wm8XBitWhcTHYuD3+0JYgP7a4fbJs7zdYJxWN2GM+nKykaqsYWZp4FAPrXwfpwHRV00w6PCGrrpg7W4ZtYHsZDRMjRF6GxIwwSEFpid8W82FUz/6K+RnW57IrfZXVvuS2Xju+S7wmQ9cS+m9uneolk20501M2ALpC6Eq6Rvnx7cXt7dTvrszRt9Rr9pQQxq9K2d9zeiekpcVHpCdEyIqo6vQ/IUbiJ1nsWPocrkJYmAHrS37iVu2YWIbegypwXeqzuAdFTxrYqREEuVSFPy9xCBlLchZuID8NpJRcIusZhQCL22D9RpOO5AzXSP0ON/21g257u/ZHeLbrRQbZkNuyRKuZ38GmvjDVo/rJF+BG4dIL9ss6gL36nqnVpLYXGMIffhPHP961gPixEI+5fUryM14cLT63M6HuUWkit9NQjPdCPePvugxlgPPp8tevRtbenjv/VOnVLMYLuedbJRAVJQjT7DDsHL2LnwQ9r15W1vfbt7rIxerseUd2ChjJs2ePS8/WjQS+ok0+nRuaxTFe6KhU0AbLBtiahb2zUySx4WrVkdE87Frt/tNed3pfZCzIOrfSwz7LvY9x/R+wCbCs1r1CKMQqvlo1yEUlGA1YghKIUvDDJNkXdWOD0CF+DfgchqyfjJ4ugEsC8wU2mW8gm437tsm6TPfNQvdzhMycmRqAYJVLi38KnKapb2KwgXCrpWoXLVmmQSdYG+2Vz+2E0zb/9t1QpID7ccgb1rK77NJY6gm3rDuIKWdB3u2z/gknje103210uacb1v+zJ3TKc76WPPY5Oc+f2+WC/g3Qo+oX0QmdpV3yNUrP2u1UEG/RFb9vQ7fc+HvXN9Hlkm36un6WQ73KQ2u3SXaYYGvz+/JezHtfHu65ncFMOGWdPLjzOg+GuW3EH2XMZsYl9ejMa7k4OQUddye3T1dOWoNeM5FaLBe6GgPmnd0LSb2eCpx0xuPPclvnU+eQJ9+wI5gm1NMbQCP7Dy5af9pnJ0g13FLRHPrxpi0p3GV5rmzxo+xRSJFVNtZuI7QSiTBy6Gx7Ow7A3tynqxMaYAsJNu1ioTeBHGPY7pkE+8AetYD4GmTMPtV/gsiVrYLG0H7+2nfSHm59+u/149c+ufd5FbL+J3vXOzwDh6b2AEP+7q8D56muI60/u2UTZRydZ1M0jT9GfAJ0OkLzcljHnuChyTt/0OffP3bf9g/8DUEsDBBQAAAAIAJKlbF02r5qJMwMAAOAFAAAZAAAAc2NyaXB0cy9leHBvcnRhcl9iYXNlcy5weX1UzW7UMBC+71OYvTgRwZUqkcOucijtIlVCVUVRD1Qry5tMugOJbWxnW7rah0I8Ai/GOD+lhYpLHNsznpnv+2bm8/mpsaiYN41hsMMKdInKMwvfOvj1g/4qYA0tec5K4xxWymfMo2bgS4cbdAw0851npXIWgvJiPp/PsLXGBVb6XbZVftvgJvvijc4e0NbYwKx2pmVWhXjDRuNL2s5mFdQM7uOBcgnqCkvIvGoobrqYsTsMWzacCmNBJymj3OqFM3e+aNCHhEKKMyzDR1AVuKRO0xmZeKAADdlHw7Qo8vzZ6T5xN7w1FTSGrzP699Bi0yi+TlltHHMUk0XXw+iLNRuSElq18KrgF93phyu5OpbvTq5WVzLP5er6/Gx1cXp+ciWoar5wCj2wa9V0sHLOuIRfmHbjIAI8vEV1l03ncWeYi/AT2IZT+qG1xRgt1i99V9d4n3BBFzxdtuiJj9viZj3iM4IsPqN9T2tCZhm/49nj+fmlPFu9/3DyaXXW4/dAyLIHcecwwIQ5H1a56wOXaLQgaGM6zxGJrvRjQhH5izi6LkTglg5814Qi3h3xYaMqI6IOeHQiDLUJbLgRcE/seeKTHh8lI/xWHb/Nk9HCEaNy8z0AWaViC/cV3gIxnr4qYtQpgBy8+PoFxD9ORqxWtDJDqm03+OvngvHXPrgkJhsVMxQZuY113vydfsa3lK1xqKZ92bkdaT9ClHHrgLCLmIH/G8CMtxAclspLErm0dAM6QH+37sFk9YBZjN7vR6BqgV5GApN0MXIulKUmqJKYOil9WRodUHeD20RondXzGrVqjvZPVH6Q+6c6PxztY7zDPP0jhfgq35EMa0q3L2AoNn5F1bXWJ3s+DQW+yPOM97BSObQdUzxkJCSqsDiOuFIp4/mCVCk63aD+mpBW/qFq9WcWPT46sTQ+kQ69QcKwjSohGcfE0jrUgV4YpkhJ9OoXptiCT3NlRllJGcuXsii4lK1CLSUnNsbRpNytVc4TrraY/sWJu+1aquwy7hwVYYWqKqnG44S/eTP0EM/Cdwt9f2SxsZHkUXxyHbzkMuT0HxdVWNEnEN2oFZaP01KJsXeVmCr7DVBLAwQUAAAACACSpWxdg13g42kGAAA/DQAAFwAAAHNjcmlwdHMvcGlsb3RvX3NpZXRlLnB5lVfrbts2FP7vp2CzH5RaVUncptscaICRuFuGtA7SrBgQGAQtURZritRISrEX5Kn6CH2xHZKy4xROgRWoIV7O/TsfTw4ODs5U3QhmKTJKKCSUQYYzy1BOZcELamFDw1ryQplTlCtpmO4oqpVVGq3hmm5A2qBGs45Tkx4cHAx43ShtEdWLhmrDErM2yRejZFJRUwk+T5RJGkFtqXQ9KLWqUUOtO0C95BUsBx+m55PLT9kt1sxIZo9/wQmu1ZwLBqvuDTE1FYIcHx3BPitLnnMmLRyRudspGLfEcrkmoDuvjt+R4fAt7Js7Lnf23xJYF+ru5/44GOhAdrUybs1XTBPTi88GBStRTbmM4tEANdkmxHSsF20N9q/cSkfxaZPSoiC0347w69e5KvhC4cSuG5a5CBPN/mm5ZkV2o1u2T8Tn/39JGCqgaj8QoVmTepedmIniAeIlommQSyWt2YsMf2zPLj+RyZCMzy/OLqYfx5cTWP59dTm9Ht9Mry+meKQpNwx9pqJlE62VjvAnrwJJhWgL4OD/UnAE9EP1U1fflDvs2OgoMVZHNA35OISUFwxwh2PnS6h/rjQj3RABrtxnQguorpJUMEOY3/ZCJgE7eZXItm7Wblc2YafjBq4nltc1VEkDVnMlMgfBVChamCjaRHyIN8eKlPwLLVTqruE41YwWxLKVjeL4lMnuGXHIu9LyR8IDBDhHy6RDXCIIxEbex8z/poR0TDtvCdn1Pdv5fnLHx5rJ5qkgBJq5nye7zdpWoGnTaWlYby5EcbJo2t6LvIXiL6B3CsB+zojDQXQUxym3rDYe7A4nkIbb5exF1u0p/yQkAhihnnNIxQjhV8sAr02Kb3EANGgIEPBLSBZwT8cgVXvUXrfATQVDgYoKXjINKWeIiq1ahfscO6+TyqX50aInKjzbDaTMtuCLsMk1b6zBzk8nn2X4KdogbRgB1hh6ROqhuwmafkKTHTcQgH7BQQ61BnwGdynklnfwUan621fJa8ehpnUo8BY0V4+UyoSLs7eRhoQ/69FoJ4itXxu3QLBn2tRUdHjyLioDIudryyAJcVqxFYgyA/h8kVV70n727avTDZ4aIEvrq+m093wh19Eq5YaYdS24XEaxz//KpX7LJXoh1DzCL/HeuvZswaSgG6IQKl9mj501+XNy9pdjHzI+u7n4PA69NUB33Fb+bqoaJiO8wrHr/XLkG7SA/ojuccMLPII0AqbhE7COK2UsHm2bQULOovghKcGw1WuHi9AKLVBjAS+errl0weeEigVU1laAn8Ch4eKc5ksmC+OaR8p0zmRe1VQvs/cU0LL/0hPFnpH33KNpTW3dihTeN3VHbPlm+COd39+CSBZatY3JHHeGwkP30E2/xad+f95yUWSBR8Mi7AfMskxAH0OBVGtH8D+cmbaGCNffnYHBvFJAG+YZjjRMMA94UvGGaXh+AElWQ3n20qXHkmGscHDSVC5Y9O746DiBn3e+gbfN7i6EMcFvw74omN7BUMMFNCc+DQdpvSy4jtgK0k/Usn8RdSuzcH5Y4nun9oHcO/MP+FRncHroJpBW2C2/e1vQBjr1qnpagX/h3m4W9NN3pILrChK4mydvIRxw+kxGgr0o6L/tGx7Pks2GYTUXgu7sQG+Dv3gG/R15XnQRJfj9+NOEnE0/XF1ObsbnY/J5fHlxPnZdhvey7yZwyHQOkyK1fC4YfnQJhrGoDwosbY03KqdAVV9YDvRdUKB8IEdE5yaq6SrqbnFHBalpruHhPIZDV1D/Pm50vd6o6nwlHXjwDALfisS/HbPXv+5x+Y8+k3s9dnZKZwdUsS9Kp42FIQ8MATWHb5hje3YGrn00nuamg9O8BaY2/cJhmLs4YWaBadLPn8yfzXo8eApXFvkal7FjzBLGy2hfqicdL4BCto77oRy/cg8lSG9Q0GjuBr3ph4sbqBvaVBInjzUuRWuqHtvwuAB9t8wLV012vxz1rXrrBGbwmofpJOREaAjrjvFF5SaBnK7x7OHUzYmPg+5HEDMQLItCj/Wtk3hqyXqKSeZuts7eDJOAhGx4koQEQXzZSQIMSPqTk+Tly6oJ0QW+aiWB5O+ANrBZ4vwII1IappQIO7LEfWr6zFxdXE5vpuj3v8bX5+Pz6TOJgXnBPdRiPfIPSSv9K1ZzY7hcbKlhAOUjfhYiBF5hQtzkTwgehb8ABv8BUEsDBBQAAAAIAJKlbF2Jj1yG7AgAAPESAAAdAAAAc2NyaXB0cy9yZWNpYmlyX2V2aWRlbmNpYXMucHmFWNtuGzkSffdXdPzCbptuWx7MXqRlACOZAAYySXaczQIjCA2qm5IY923IbtlykI+ZD9infZvX/NieItm6JJpNHqImWVWs66miT09Pb2pZ6icZ/Xr7LlKPbWM6WTQ2srrGsjNSmRLLWkdVU+iFzqWJGqOXGmzKptFdUzbRGiIKmesv/63T09PTE12RnEiaZSuNVTy3a76SdlXqOdcN/2ibmj/pdqFLdbIwTRW1sqPDKDC+w3IQUvdVu4kkVGg9rb0vlTR1WqnO6NwOPItRZvPGKC7zvDcy34RlJbtupR5shpXJG7Xgpskz2ecD+VoZuVRZa1SurW7qsJ839aJ3a0gw+vHkxeubu7uf7sSUdX3VGMZZqetFk+uuwbeyHZSTbHZSqEVkmgcbF7KTydiorjd1VGrbxfBD+lLn3S9KFsrEuknvILpe3r51xGmh8qZQMeu7xcXfLqxesgT/nMTWNHM516XutLKxMcn4JNKLqG66yJixkdqq6IMse/WTMY2J2TujCp3n0F9ZxCf/8h9pWXIS3auNFdPYTFmLgKm6U2zGsZImX+l1w2ZJtGhMZCLE35iZu6RUdWxVFxNvkjwTtHbf37u36NsSGVP4q0s5V6UVn7z/srqpf4bvOp2z8RWcuanaVZNvoNB4xBlO39++Jpp/Zt61JRtf8+B7x7D1PtEP/h9ff55sRN2m0hi5iaf+0ikszEtpVWYUBM1mB0Ymk3aPY7ooG9k5D2XsPA8OyYk2pMBX3NtAQIa2C13rTsVtksqyjBMUSxS3/7jCst4My+ejvWXgBHWOSgNnavsqHiV8xGXXlGKkLn487umQEQWSCWVQr7/8Tgvn65B1G96euPQJxRJjgzIHyV4IsnhZyUfcNWlW5AG1UfEPyXQzm1D260LVuQJZoKECUnknHLPYnERzXVtiq2ByBZ3jHdfZiCy23aZVsa67hP89mSgIu0onuSzFFIlFPpw7H8p6qWIwQLFIPwonVsxpscDae4rOIpS28NEJukz14wxAIOs4SZzO2+NBkQMKaHAeKCA37J7JuY0h+YKYElIvlW2r6iJGIncxtBFzXuoKUc2Qcsroxoj55ehq2LR96zfj+fkooYNawGa6giKZJFtIEvhwwKJl/SSFuxDBAhiJIev2sSluVtMxz2e8dT9JMhQjfN7X+rd+S5EkQlxHSHQVvUHl7TLW+/aHBP6W7XDJn0De0evCnrfk+dV37ghp5xxXO5zYJEDg3DTZYhQ8PwA15SIyiQdYmF4h4a9nAx4L5tgYf1JgLvTaKSqu9r0ZrDnA+yAVZNU2Wb5pATuiudHKZFVfAoYcPsBOGdjiuL1oVsnZ2XUSSnJIGY5EykZXAj+cooUGlDl2gRWX7f5Gy13/eMq2DUV83VqOeyJJUf3UNRKOnNRz2OiYZZn4mpZowZsnFXvoVpY3fRfaAr5S9QhmGx+Djn9ZGfW1jNDMW9VJ9Fi1lhMgUQRC28yNsrnRc1VHhLGmUgFT4L8C+s0mtulNrvynUrX49NnXc1CFsmLQigo3kB/UVWg4ItCltawUtyt5/eNfRJgVUr8cDEyB3UU2R4eAVUm6Uo+FXipyEBVR9KC7VRSmivRX3b7C78Ca0ADx5DAE3nlKIaID6VHn0CBEidK3aC0k2FlG6pFZFrOGKuInp68Pj8cmEkxzS0wHiTt+Jli71w6zYUxq6hSTABsjDzoN3zt2TEroxoL4UmPRN7uYXTI+SqZXswngWwpPcc4uER4kLMa0lCYpNtxeuVCim0DNff2O2PhKgj86lBN97OuuiWTEzp0JTqwRdJSiHgoLoykCMd0DMDVGuDEn7DqeZIJUFt/MKl5D4HGnu42g1ophUpWYNjgBJdZWVbosMT4l3JvpeXyAv86Hgxv382DwxHCVCxjy0weITmg1HY5nz4TnPOKhlxh6gUTWDzK6zpsKQ6qeY+gdw0OYNuJBTOLvjQ7CeXhRuMedVGK/GU+Mal1KCZMuMWOxXZIw/ulz4jcHCGVbEwe2SFsXcwfJsi4i6mbVdMcxuxhIk+cYJf56xNafiTZ6NSIAyBvYCr3HB2lApsMUuzcmrfcnSFci6904VbkOH0onMO9VDw2Tg8ihfKjzb68ZvibrnbOok6M14X+64ABLBkUGtm8a8LYFrfc9s+sk2B6+kYLB5hXhN5v3Fo22kIy8zi6H5SUje1Ch59uyZL45xqzVZQPs8Ax+8efkrMHgGqDBoCWDOMebKkQ6YO6Btb52xH4VhfIR31bTgoxwpnDTb0GEH+BSQF2fo9w7XFpRcWpjO99WyW7UDXodSaY3TbSSm6g9eAeo/SciFcpX2DOhxmPUbz26sRqeoQNx1ZZAHHIItbXqvtAm9mZY8d70yrUm4hDOPbABYgW7eXPz+vbu9i57+dPdi19u372//fA2+4C9lzcvbt++YXzRk11WhO7ECfRpfhbBOD/fSW8Dnn0f9kzoO02v5kISbstIfuxth5pBAcFc6i4Id6EoZr5COPkFNsol+op0LljoZW/kzkPzL3/41HL+CSF0JyGfOLvTriPDR6hR6YUYzBB4kEMRV2o+WPRe9zoRf6cqTKgSo5n0iqDbl1/+cA1AmbV01HmuQAqhK5Xft0CBzjJ628Rw+iWDS1TrOpdrOUn6YGj07dRjF7sOUeCZDqx3ceCqtr1RmbS51gK9BslH7kB7u+Z45zQPWS1rf0A5tShlJ6bfy+39vPZpjR36pQcspTaW9Ivl2dmn+zHJCKnMZtN7j1H3VIZTBmzd4QDf1T528xz/Hx0L6ZXv5j42+7z/SvbZArxzE4j313Dz1z0/SRskRcweGK/VA56wQBjmppMFweCDGP5E8G/yr4kXfKFVWRASW+HaOTkLIwFa7YMPwsr/MWG7dk2ZqHbPQBrEg5rJCSo4y0hglgnBMsygus4yRq9CMfzFJr0xy75CwN7RysSFGwjbjgbQLCvwSMkAwqksikwGyphdXGCkgl3YsYKdM06vP0EzEXelTc9GV69HOK1z0/9hAXKlTjViw/w3aQ2BHXsRitZNw64i7Zjx7WicQiUuUy8eufY/UEsBAhQDFAAAAAgAkqVsXeaVoXLCCwAA0BgAACAAAAAAAAAAAAAAAIABAAAAAHNjcmlwdHMvZXZhbHVhY2lvbl9kZXNhcnJvbGxvLnB5UEsBAhQDFAAAAAgAkqVsXTavmokzAwAA4AUAABkAAAAAAAAAAAAAAIABAAwAAHNjcmlwdHMvZXhwb3J0YXJfYmFzZXMucHlQSwECFAMUAAAACACSpWxdg13g42kGAAA/DQAAFwAAAAAAAAAAAAAAgAFqDwAAc2NyaXB0cy9waWxvdG9fc2lldGUucHlQSwECFAMUAAAACACSpWxdiY9chuwIAADxEgAAHQAAAAAAAAAAAAAAgAEIFgAAc2NyaXB0cy9yZWNpYmlyX2V2aWRlbmNpYXMucHlQSwUGAAAAAAQABAAlAQAALx8AAAAA')
assert hashlib.sha256(payload).hexdigest()=='4a22f9708b246845bc97aaaca24d7e7c5c3fb8d780b9534db8be35fc6a9bf383'
HELPERS.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
 for name in z.namelist():
  p=(HELPERS/name).resolve()
  assert p.is_relative_to(HELPERS.resolve())
  p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(z.read(name))
import sklearn
print('Receptor preparado; scikit-learn',sklearn.__version__)


## Ejecutar la cola sin preguntas intermedias
No iniciar esta celda mientras haya otro entrenamiento. Mantiene un registro por bloque y genera un ZIP incluso si un bloque falla. Los checkpoints se guardan en sus carpetas originales de E2.

In [ ]:
import time
lock=ROOT/'EJECUCION_ACTIVA.json'
with lock.open('x') as f:json.dump({'pid':os.getpid(),'host':platform.node()},f)
estado_path=ROOT/'estado_cola.json'
estado={'estado':'EN_EJECUCION','bloques':[],'reserva_evaluada':False}
def guardar():
    tmp=estado_path.with_suffix('.tmp');tmp.write_text(json.dumps(estado,ensure_ascii=False,indent=2));tmp.replace(estado_path)
def correr(nombre,cmd):
    log=ROOT/(nombre+'_'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%f')+'.txt')
    estado['bloque_actual']=nombre;guardar();print('INICIANDO',nombre,flush=True)
    with log.open('w') as f:
        proc=subprocess.Popen([str(x) for x in cmd],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in proc.stdout:print(line,end='',flush=True);f.write(line);f.flush()
            rc=proc.wait()
        except BaseException:
            proc.send_signal(__import__('signal').SIGINT)
            try:proc.wait(timeout=120)
            except subprocess.TimeoutExpired:
                proc.terminate();proc.wait(timeout=60)
            raise
    estado['bloques'].append({'nombre':nombre,'codigo_salida':rc,'log':str(log)});guardar()
    if rc:raise RuntimeError('Falló '+nombre+'. Evidencia conservada; revisar '+str(log))
def fase(codigo,datos,salida,f,extra=False):
    c=[sys.executable,'-u',codigo/'scripts/adicionales_e2.py','--datos',datos,'--salida',salida,'--fase',f]
    if extra:c+=['--autorizar','EJECUTAR_TRES_NUEVOS_V1']
    return c
def empaquetar():
    target=DRIVE/'NuCLS_E2_COLA_UNICA_V1_RESULTADOS.zip';tmp=target.with_suffix('.tmp')
    bases=[ROOT,SALIDA_ANTERIOR,SALIDA_NUEVOS,DRIVE/'NuCLS_E2_REFERENCIAS_SIMPLES']
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED) as z:
        for root in bases:
            for p in sorted(root.rglob('*')):
                if p.is_file() and not p.is_symlink() and p.suffix in ('.json','.csv','.txt','.png','.npy') and p.name!='EJECUCION_ACTIVA.json':z.write(p,root.name+'/'+str(p.relative_to(root)))
    tmp.replace(target);print('ZIP DE AVANCE:',target,flush=True)
try:
    # Search runner validates and skips complete trials; no use of a loose status marker.
    correr('01_completar_busqueda',fase(CODIGO_ANTERIOR,DATOS_ANTERIORES,SALIDA_ANTERIOR,'busqueda'))
    if not (SALIDA_NUEVOS/'perfil.json').exists() or len(json.loads((SALIDA_NUEVOS/'perfil.json').read_text()))!=3:correr('02_perfil_tres',fase(CODIGO_NUEVOS,DATOS_NUEVOS,SALIDA_NUEVOS,'perfil',True))
    correr('03_busqueda_tres',fase(CODIGO_NUEVOS,DATOS_NUEVOS,SALIDA_NUEVOS,'busqueda',True));empaquetar()
    correr('04_piloto_siete',[sys.executable,'-u',HELPERS/'scripts/piloto_siete.py','--codigo',CODIGO_ANTERIOR,'--datos',DATOS_ANTERIORES,'--salida',SALIDA_ANTERIOR]);empaquetar()
    correr('05_piloto_tres',fase(CODIGO_NUEVOS,DATOS_NUEVOS,SALIDA_NUEVOS,'piloto',True));empaquetar()
    correr('06_referencias',[sys.executable,'-u',CODIGO_ANTERIOR/'scripts/baselines_simples.py','--datos',DATOS_ANTERIORES,'--salida',DRIVE/'NuCLS_E2_REFERENCIAS_SIMPLES']);empaquetar()
    correr('07_robustez_atribuciones_latencia',[sys.executable,'-u',HELPERS/'scripts/evaluacion_desarrollo.py','--codigo',CODIGO_ANTERIOR,'--nuevos',CODIGO_NUEVOS,'--datos',DATOS_ANTERIORES,'--drive',DRIVE,'--salida',ROOT/'evaluacion_desarrollo']);empaquetar()
    correr('08_exportar_bases',[sys.executable,'-u',HELPERS/'scripts/exportar_bases.py','--indice',INDICE,'--salida',DRIVE/'NuCLS_E2_BASES_66_EVIDENCIAS.zip'])
    empaquetar()
    carpeta=ROOT/('analisis_'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S'))
    correr('09_metricas',[sys.executable,'-u',HELPERS/'scripts/recibir_evidencias.py','--zip',DRIVE/'NuCLS_E2_BASES_66_EVIDENCIAS.zip',DRIVE/'NuCLS_E2_COLA_UNICA_V1_RESULTADOS.zip','--salida',carpeta])
    estado['estado']='COLA_EXPLORATORIA_COMPLETADA'
    estado['pendientes_no_automatizados']=['Dimensionar y ejecutar benchmark final después del piloto','Replicación numérica del artículo guía con datos/protocolo externos','Techo multi-rater y ruido: faltan anotaciones enlazadas','Inferencia confirmatoria, calibración independiente y reproducción externa']
except BaseException as exc:
    estado['estado']='DETENIDA_CON_EVIDENCIA_CONSERVADA';estado['error']=str(exc);raise
finally:
    guardar()
    try:empaquetar()
    finally:lock.unlink(missing_ok=True)
print(json.dumps(estado,ensure_ascii=False,indent=2))


## Al regresar
Si aparece `COLA_EXPLORATORIA_COMPLETADA`, descargar `NuCLS_E2_COLA_UNICA_V1_RESULTADOS.zip` y `NuCLS_E2_BASES_66_EVIDENCIAS.zip` desde Drive. Guardar este notebook con salidas. No incluye pesos; permanecen en Drive.

Si se interrumpe, el registro identifica el último bloque y las corridas completas se verifican antes de omitirlas. No borrar bloqueos de otros entornos. Los mapas y evaluaciones son exploratorios en validación; no se presentan como resultados de reserva ni como cumplimiento total de la guía.
